# Introduction to Python

**June 22, 2026**

Run a code cell with **Shift+Enter**. Change the examples and run them again; Python errors are useful evidence about what the computer understood.

### Contents
1. values, variables, and types
2. Unicode strings and word lists
3. loops, conditions, and functions
4. transliteration between Devanagari and IAST

## 1. Python Code
A string is text enclosed in quotation marks. `print` displays a value.

In [1]:
verse = "ॐ अग्निमीळे पुरोहितं यज्ञस्य देवमृत्विजम्"
print(verse)

ॐ अग्निमीळे पुरोहितं यज्ञस्य देवमृत्विजम्


## 2. Values, variables, and types
A variable is a name bound to a value. `type` reports the kind of value currently stored.

In [2]:
text_name = "ऋग्वेद"       # str: text
mandala = 1                # int: whole number
confidence = 0.95          # float: decimal number
is_vedic = True            # bool: True or False

for value in [text_name, mandala, confidence, is_vedic]:
    print(value, type(value).__name__)

ऋग्वेद str
1 int
0.95 float
True bool


### How a cell executes
Python evaluates statements from top to bottom. A literal creates a value, `=` binds a value to a name, and a function call can consume values and return a new one. Names remain in the notebook runtime after the cell finishes, so execution order matters. Editing a cell does nothing until you run it again.

In [3]:
text = "रामः"          # create a str and bind it to text
count = len(text)       # len returns an int
print(count, type(count).__name__)

4 int


## 3. Unicode strings
Python 3 stores Unicode text directly. One displayed akṣara may contain several Unicode code points, so `len(text)` is not an akṣara counter.

In [4]:
for text in ["क", "कि", "क्ष"]:
    code_points = [f"U+{ord(ch):04X}" for ch in text]
    print(repr(text), "len =", len(text), code_points)

'क' len = 1 ['U+0915']
'कि' len = 2 ['U+0915', 'U+093F']
'क्ष' len = 3 ['U+0915', 'U+094D', 'U+0937']


In [5]:
# The mismatch between code-point count and akṣara count matters for syllable analysis.
for word, note in [("कि", "1 akṣara"), ("कृष्ण", "2 akṣaras"), ("महाभारत", "5 akṣaras")]:
    pts = [f"U+{ord(c):04X}" for c in word]
    print(f"{word:12} len={len(word)}  code points={pts}  ({note})")
# A metre analyser needs akṣara count. len() gives neither akṣaras nor mātrās.

कि           len=2  code points=['U+0915', 'U+093F']  (1 akṣara)
कृष्ण        len=5  code points=['U+0915', 'U+0943', 'U+0937', 'U+094D', 'U+0923']  (2 akṣaras)
महाभारत      len=7  code points=['U+092E', 'U+0939', 'U+093E', 'U+092D', 'U+093E', 'U+0930', 'U+0924']  (5 akṣaras)


### Inspect Unicode more closely
`unicodedata` exposes properties of individual code points. UTF-8 then converts a Python `str` into bytes for storage or transfer. Neither level supplies linguistic segmentation.

In [6]:
import unicodedata as ud

text = "कि"
for ch in text:
    print(f"U+{ord(ch):04X}", ud.name(ch), ud.category(ch))

utf8 = text.encode("utf-8")
print("code points:", len(text), "bytes:", len(utf8), utf8)

U+0915 DEVANAGARI LETTER KA Lo
U+093F DEVANAGARI VOWEL SIGN I Mc
code points: 2 bytes: 6 b'\xe0\xa4\x95\xe0\xa4\xbf'


In [7]:
# These strings look identical but initially contain different code points.
precomposed = "ā"        # U+0101
decomposed = "a\u0304"  # U+0061 + COMBINING MACRON

print(precomposed, decomposed)

ā ā


In [8]:
print(precomposed == decomposed)
print([f"U+{ord(ch):04X}" for ch in precomposed])
print([f"U+{ord(ch):04X}" for ch in decomposed])
print(ud.normalize("NFC", precomposed) == ud.normalize("NFC", decomposed))

False
['U+0101']
['U+0061', 'U+0304']
True


In [9]:
try:
    import regex
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "regex"])
    import regex

for sample in ["कि", "क्ष", "रामोऽस्ति"]:
    clusters = regex.findall(r"\X", sample)
    print(sample, "→", clusters)

# \X finds Unicode extended grapheme clusters, not grammatical words or sandhi splits.

कि → ['कि']
क्ष → ['क्ष']
रामोऽस्ति → ['रा', 'मो', 'ऽ', 'स्ति']


## 4. Useful string operations
`strip` removes surrounding whitespace. `split` uses written whitespace; it does **not** reverse sandhi or split compounds.

In [10]:
sentence = "  रामः वनं गच्छति  "
clean_sentence = sentence.strip()
written_words = clean_sentence.split()

print(clean_sentence)
print(written_words)
print("written segments:", len(written_words))

रामः वनं गच्छति
['रामः', 'वनं', 'गच्छति']
written segments: 3


## 5. Lists and loops
A list keeps values in order. A loop performs the indented block once for each value.

In [11]:
padas = ["धर्मक्षेत्रे", "कुरुक्षेत्रे", "समवेता", "युयुत्सवः"]

for index, pada in enumerate(padas, start=1):
    print(index, pada, len(pada))

1 धर्मक्षेत्रे 12
2 कुरुक्षेत्रे 12
3 समवेता 6
4 युयुत्सवः 9


## 6. Dictionaries and conditions
A dictionary maps a key to a value. A condition chooses which block to run.

In [12]:
glosses = {
    "रामः": "Rāma (nominative singular)",
    "वनम्": "forest (accusative singular)",
    "गच्छति": "goes",
}

query = "वनम्"
if query in glosses:
    print(glosses[query])
else:
    print("No entry for", query)

forest (accusative singular)


## 7. Functions
A function gives a name to a computation and makes its inputs explicit.

In [13]:
def written_word_count(text):
    """Count whitespace-separated segments in an edited text."""
    return len(text.strip().split())

sample = "अग्निमीळे पुरोहितं यज्ञस्य देवमृत्विजम्"
print(written_word_count(sample))

4


### Optional: compact Python idioms
Comprehensions build collections from an iterable. They are useful when the transformation is short and clear; use an ordinary loop when several steps or explanations are needed.

In [14]:
# Dictionary comprehension: written segment → code-point count
code_point_counts = {pada: len(pada) for pada in padas}

# List comprehension with a condition
long_segments = [pada for pada in padas if len(pada) > 8]

print(code_point_counts)
print(long_segments)

{'धर्मक्षेत्रे': 12, 'कुरुक्षेत्रे': 12, 'समवेता': 6, 'युयुत्सवः': 9}
['धर्मक्षेत्रे', 'कुरुक्षेत्रे', 'युयुत्सवः']


## 8. Transliteration

Transliteration changes representation; it does not reverse sandhi or perform grammatical analysis.

Four schemes are shown: ITRANS (ASCII input scheme, widely used in online Sanskrit communities), IAST (scholarly standard), ISO 15919, and WX (used in several NLP toolchains including Sanskrit Heritage Reader).

In [15]:
try:
    import indic_transliteration
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "indic-transliteration"] )

In [16]:
from indic_transliteration import sanscript
from indic_transliteration.sanscript import transliterate

deva = "रामः वनं गच्छति"
print("Devanagari:", deva)
for name, scheme in [
    ("ITRANS", sanscript.ITRANS),    # ASCII input scheme; ā ī ū as aa ii uu
    ("IAST",   sanscript.IAST),      # older western scholarly standard with diacritics
    ("ISO",    sanscript.ISO),       # ISO 15919; modern international standard for the romanisation of Indic scripts
    ("WX",     sanscript.WX),        # used in several Sanskrit NLP tools
]:
    print(f"{name:8}: {transliterate(deva, sanscript.DEVANAGARI, scheme)}")

Devanagari: रामः वनं गच्छति
ITRANS  : rAmaH vanaM gachChati
IAST    : rāmaḥ vanaṃ gacchati
ISO     : rāmaḥ vanaṁ gacchati
WX      : rAmaH vanaM gacCawi


### What does the transliteration map?

Sanskrit is the **language**; Devanagari is one **script** used to write it; IAST, ISO 15919, ITRANS, WX are some of the **transliteration schemes**. Transliteration systematically changes the written representation while aiming to preserve relevant distinctions. It is not translation, and it is not automatically a phonetic transcription of an utterance.

Devanagari is an abugida: a consonant sign ordinarily carries an inherent `a`; a vowel sign replaces that vowel, and virāma suppresses it. Transliteration must therefore interpret consonants, vowel signs, virāma, and other marks rather than replace each displayed glyph independently.

In [17]:
# Inherent vowel, vowel signs, virāma, conjunct, anusvāra, and visarga
for unit in ["क", "का", "कि", "क्", "क्ष", "कं", "कः"]:
    iast = transliterate(unit, sanscript.DEVANAGARI, sanscript.IAST)
    print(f"{unit:3} -> {iast}")

क   -> ka
का  -> kā
कि  -> ki
क्  -> k
क्ष -> kṣa
कं  -> kaṃ
कः  -> kaḥ


### Schemes serve different purposes
| Scheme | Typical strength | Practical cost |
|---|---|---|
| IAST | Older standard in Sanskrit scholarship | Requires Unicode diacritics |
| ISO 15919 | Standardized across Indic scripts | Some conventions differ from IAST |
| ITRANS | Convenient ASCII keyboard input | One Sanskrit unit may require several characters, multiple valid representations |
| WX | One-character ASCII mapping used in several Indian NLP systems | Case conventions must be learned |

A string is meaningful only together with its declared scheme. Do not guess the source scheme from appearance alone.

In [18]:
comparison_schemes = [
    ("IAST",   sanscript.IAST),
    ("ISO",    sanscript.ISO),
    ("ITRANS", sanscript.ITRANS),
    ("WX",     sanscript.WX),
]

for example in ["कृष्णः", "संस्कृतम्", "रामोऽस्ति", "ज्ञानम्"]:
    print("\nDevanagari:", example)
    for name, scheme in comparison_schemes:
        converted = transliterate(example, sanscript.DEVANAGARI, scheme)
        print(f"{name:8}: {converted}")


Devanagari: कृष्णः
IAST    : kṛṣṇaḥ
ISO     : kr̥ṣṇaḥ
ITRANS  : kRRiShNaH
WX      : kqRNaH

Devanagari: संस्कृतम्
IAST    : saṃskṛtam
ISO     : saṁskr̥tam
ITRANS  : saMskRRitam
WX      : saMskqwam

Devanagari: रामोऽस्ति
IAST    : rāmo'sti
ISO     : rāmō'sti
ITRANS  : rAmo.asti
WX      : rAmo'swi

Devanagari: ज्ञानम्
IAST    : jñānam
ISO     : jñānam
ITRANS  : j~nAnam
WX      : jFAnam


### Round-trip testing
A useful engineering check is to convert source → target → source and compare the result. Equality shows that this example survived this conversion path; it does not prove that every Vedic accent, punctuation mark, editorial symbol, or script-specific distinction is supported. Preserve the original text and record the source scheme and conversion options.

In [19]:
for original in ["रामः वनं गच्छति", "रामोऽस्ति", "संस्कृतम्"]:
    roman = transliterate(original, sanscript.DEVANAGARI, sanscript.ISO)
    restored = transliterate(roman, sanscript.ISO, sanscript.DEVANAGARI)
    print(original)
    print("  ISO:      ", roman)
    print("  restored:  ", restored)
    print("  same text?", original == restored)

रामः वनं गच्छति
  ISO:       rāmaḥ vanaṁ gacchati
  restored:   रामः वनं गच्छति
  same text? True
रामोऽस्ति
  ISO:       rāmō'sti
  restored:   रामोऽस्ति
  same text? True
संस्कृतम्
  ISO:       saṁskr̥tam
  restored:   संस्कृतम्
  same text? True


In [20]:
for original in ["रामः वनं गच्छति", "रामोऽस्ति", "संस्कृतम्"]:
    for scheme in [sanscript.ITRANS, sanscript.HK, sanscript.WX, sanscript.IAST]:
        scheme_sent = transliterate(original, sanscript.DEVANAGARI, scheme)
        restored = transliterate(scheme_sent, scheme, sanscript.DEVANAGARI)
        print(f"{scheme}:", original, "->", scheme_sent, "->", restored)
        print("  same text?", original == restored)

    print("---")

itrans: रामः वनं गच्छति -> rAmaH vanaM gachChati -> रामः वनं गच्छति
  same text? True
hk: रामः वनं गच्छति -> rAmaH vanaM gacchati -> रामः वनं गच्छति
  same text? True
wx: रामः वनं गच्छति -> rAmaH vanaM gacCawi -> रामः वनं गच्छति
  same text? True
iast: रामः वनं गच्छति -> rāmaḥ vanaṃ gacchati -> रामः वनं गच्छति
  same text? True
---
itrans: रामोऽस्ति -> rAmo.asti -> रामोऽस्ति
  same text? True
hk: रामोऽस्ति -> rAmo'sti -> रामोऽस्ति
  same text? True
wx: रामोऽस्ति -> rAmo'swi -> रामोऽस्ति
  same text? True
iast: रामोऽस्ति -> rāmo'sti -> रामोऽस्ति
  same text? True
---
itrans: संस्कृतम् -> saMskRRitam -> संस्कृतम्
  same text? True
hk: संस्कृतम् -> saMskRtam -> संस्कृतम्
  same text? True
wx: संस्कृतम् -> saMskqwam -> संस्कृतम्
  same text? True
iast: संस्कृतम् -> saṃskṛtam -> संस्कृतम्
  same text? True
---


### Combining functions into a workflow
`convert_from_devanagari` performs one conversion and returns the result. `show_representations` calls it several times, so the conversion rule is written in only one place. The whitespace cleanup below changes layout only; it does not perform Sanskrit word segmentation.

In [21]:
def convert_from_devanagari(text, target_scheme):
    clean_text = " ".join(text.split())
    return transliterate(
        clean_text,
        sanscript.DEVANAGARI,
        target_scheme,
    )


def show_representations(text):
    representations = {
        "Devanagari": " ".join(text.split()),
        "IAST": convert_from_devanagari(text, sanscript.IAST),
        "SLP1": convert_from_devanagari(text, sanscript.SLP1),
        "WX": convert_from_devanagari(text, sanscript.WX),
    }
    for scheme_name, converted_text in representations.items():
        print(f"{scheme_name:10}: {converted_text}")


show_representations("  रामो विग्रहवान् धर्मः  " )

Devanagari: रामो विग्रहवान् धर्मः
IAST      : rāmo vigrahavān dharmaḥ
SLP1      : rAmo vigrahavAn DarmaH
WX        : rAmo vigrahavAn XarmaH


### Script conversion is not grammatical analysis
Unicode lets Python store and manipulate `रामोऽस्ति`. A transliterator can render the same sequence as `rāmo'sti`. Neither operation recovers the sandhi split `रामः + अस्ति`, the nominal features of `रामः`, or the verbal analysis of `अस्ति`. Those require linguistic rules, lexica, annotated data, or models.

Try the same example in [Aksharamukha](https://www.aksharamukha.com/): choose Devanagari as the source, IAST as the target, enter `रामोऽस्ति`, and inspect how the avagraha is represented. Compare the result with the Python output above. Then ask what neither conversion has supplied: the split `रामः + अस्ति` and its grammatical analysis. Check the conversion options and preserve the original text.

## 9. Utilities for Sanskrit alphabet with `sanskrit-text` library
Three utilities for Devanagari text: `get_syllables_word` segments a word into syllable-like akṣara units, `split_varna` performs varṇa-viccheda, and `get_ucchaarana` reports dimensions of uccāraṇa-sthāna and uccāraṇa-yatna.

The general `get_syllables` function preserves the hierarchy **text -> lines -> words -> units**. For a single-word demonstration, `get_syllables_word` gives the level we need directly.

In [22]:
try:
    import sanskrit_text as skt
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "sanskrit-text"])
    import sanskrit_text as skt

In [23]:
# Work at the word level so len() counts the returned units, not outer nesting levels.
for word in ["अ", "कृष्ण", "महाभारत"]:
    aksharas = skt.get_syllables_word(word)
    print(f"{word}: {len(aksharas)} akṣara(s)  →  {aksharas}")

अ: 1 akṣara(s)  →  ['अ']
कृष्ण: 2 akṣara(s)  →  ['कृ', 'ष्ण']
महाभारत: 5 akṣara(s)  →  ['म', 'हा', 'भा', 'र', 'त']


In [24]:
# technical=False gives independent vowels rather than internal technical markers.
# flat=True removes the text/line/word nesting for these single-word examples.
for word in ["पचति", "कृष्णः", "ज्ञानम्"]:
    varnas = skt.split_varna(word, technical=False, flat=True)
    print(f"{word} = {varnas}")

पचति = ['प्', 'अ', 'च्', 'अ', 'त्', 'इ']
कृष्णः = ['क्', 'ऋ', 'ष्', 'ण्', 'अः']
ज्ञानम् = ['ज्', 'ञ्', 'आ', 'न्', 'अ', 'म्']


In [25]:
# Dimensions: 0 = sthāna, 1 = ābhyantara-prayatna, 2 = bāhya-prayatna.
def consonant_feature(char, dimension):
    # Result shape: text → line → word → (varṇa, feature).
    entries = skt.get_ucchaarana(char, dimension=dimension)[0][0]
    return entries[0][1]  # feature of the consonant, not its inherent अ

for char in ["क", "च", "ट", "त", "प", "य", "र", "ल", "व", "श", "ष", "स", "ह"]:
    sthana = consonant_feature(char, 0)
    abhyantara = consonant_feature(char, 1)
    bahya = consonant_feature(char, 2)
    print(f"{char}: sthāna={sthana}; ābhyantara={abhyantara}; bāhya={bahya}")

क: sthāna=कण्ठः; ābhyantara=स्पृष्टः; bāhya=विवारः श्वासः अघोषः अल्पप्राणः च
च: sthāna=तालु; ābhyantara=स्पृष्टः; bāhya=विवारः श्वासः अघोषः अल्पप्राणः च
ट: sthāna=मूर्धा; ābhyantara=स्पृष्टः; bāhya=विवारः श्वासः अघोषः अल्पप्राणः च
त: sthāna=दन्ताः; ābhyantara=स्पृष्टः; bāhya=विवारः श्वासः अघोषः अल्पप्राणः च
प: sthāna=ओष्ठौ; ābhyantara=स्पृष्टः; bāhya=विवारः श्वासः अघोषः अल्पप्राणः च
य: sthāna=तालु; ābhyantara=ईषत्स्पृष्टः; bāhya=संवारः नादः घोषः अल्पप्राणः च
र: sthāna=मूर्धा; ābhyantara=ईषत्स्पृष्टः; bāhya=संवारः नादः घोषः अल्पप्राणः च
ल: sthāna=दन्ताः; ābhyantara=ईषत्स्पृष्टः; bāhya=संवारः नादः घोषः अल्पप्राणः च
व: sthāna=दन्तौष्ठम्; ābhyantara=ईषत्स्पृष्टः; bāhya=संवारः नादः घोषः अल्पप्राणः च
श: sthāna=तालु; ābhyantara=ईषद्विवृतः; bāhya=विवारः श्वासः अघोषः महाप्राणः च
ष: sthāna=मूर्धा; ābhyantara=ईषद्विवृतः; bāhya=विवारः श्वासः अघोषः महाप्राणः च
स: sthāna=दन्ताः; ābhyantara=ईषद्विवृतः; bāhya=विवारः श्वासः अघोषः महाप्राणः च
ह: sthāna=कण्ठः; ābhyantara=ईषद्विवृतः; bāhya=संवारः नादः घोष

## 10. A small text-processing function
This combines a function, a list, a loop, and transliteration without pretending that whitespace tokenization solves Sanskrit segmentation.

In [26]:
def show_segments(text_devanagari):
    segments = text_devanagari.strip().split()
    for index, segment in enumerate(segments, start=1):
        segment_iast = transliterate(segment, sanscript.DEVANAGARI, sanscript.IAST)
        print(f"{index:>2}. {segment:<12} {segment_iast}")

show_segments("मा निषाद प्रतिष्ठां त्वमगमः शाश्वतीः समाः")

 1. मा           mā
 2. निषाद        niṣāda
 3. प्रतिष्ठां   pratiṣṭhāṃ
 4. त्वमगमः      tvamagamaḥ
 5. शाश्वतीः     śāśvatīḥ
 6. समाः         samāḥ


## Exercises
1. Change the text passed to `show_segments`.
2. Write `to_iso_list(words)` and `to_wx_list(words)`, returning a list of ISO and WX strings.
3. Count how often each written segment occurs in a short passage.
4. Explain why `"रामोऽस्ति".split()` cannot recover `रामः + अस्ति`.

In [27]:
def to_iso_list(words):
    # Replace the empty list with your solution.
    return []

def to_wx_list(words):
    # Replace the empty list with your solution.
    return []

to_iso_list(["राम", "सीता", "लक्ष्मण"])
to_wx_list(["राम", "सीता", "लक्ष्मण"])

[]

## Summary
- Python values have types; variables are names for values.
- A string is a sequence of Unicode code points, not a sequence of linguistic akṣaras; `len()` is not a syllable counter.
- `str` holds Unicode text; `bytes` holds an encoded representation such as UTF-8.
- Normalize Unicode before exact comparison when equivalent text may use different code-point sequences.
- Lists, loops, dictionaries, conditions, and functions are enough to begin small text-processing experiments.
- Transliteration changes representation. Linguistic analysis requires additional rules, data, or models.
- As an extension, `sanskrit-text` provides syllabification, varṇa decomposition, and articulatory-feature utilities. Its general text functions return nested structures that preserve lines and words.

## More Resources

- **Starting from zero:** [Python for Beginners](https://www.python.org/about/gettingstarted/)
- **Learning the language systematically:** [The official Python tutorial](https://docs.python.org/3/tutorial/) — best once the basic programming ideas are familiar
- **Working in notebooks:** [Welcome to Colab](https://colab.research.google.com/notebooks/intro.ipynb)

Keep modifying this notebook with small Sanskrit examples; running and changing code is more useful than reading syntax alone.